# Synthetic Signal Generation + OLS Analysis

## Generative model

$$y_i(s) = x_i(s) + e_i(s), \qquad x_i(s) = A_i\,\phi\!\left(\frac{s-\mu}{bw}\right)$$

where $e_i(s)$ is Gaussian-smoothed noise.  
We use the **unnormalized** Gaussian bump throughout, so $A_i$ controls peak height directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

np.random.seed(42)

# ── Global parameters ─────────────────────────────────────
n_instances    = 1000   # number of subjects
n_pixels       = 512    # length of 1-D "image"
s              = np.arange(n_pixels)
mu             = n_pixels // 2   # bump centre  (pixel 256)
bw_signal      = 20              # bump width (SD of Gaussian shape)
noise_kernel_sd = 5              # spatial smoothness of noise
sigma_A        = 2.0             # SD of amplitude distribution


## 1 · Gaussian signal shapes

$$\phi_{\text{unnorm}}(s) = \exp\!\left(-\frac{(s-\mu)^2}{2\,bw^2}\right)$$

Peak is always 1; $A_i$ scales the height directly.


In [ ]:
def gaussian_unnormalized(s, mu, bw):
    return np.exp(-((s - mu) ** 2) / (2 * bw ** 2))

def gaussian_normalized(s, mu, bw):
    return (1 / (bw * np.sqrt(2 * np.pi))) * np.exp(-((s - mu) ** 2) / (2 * bw ** 2))

phi = gaussian_unnormalized(s, mu, bw_signal)   # the one we will use

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(s, phi)
axes[0].set(title="Signal shape φ(s)  [unnormalised]",
            xlabel="s (pixel)", ylabel="φ(s)")
axes[0].axvline(mu, color="gray", ls="--", lw=0.8, label=f"centre μ={mu}")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for bw in [5, 10, 20, 50]:
    axes[1].plot(s, gaussian_unnormalized(s, mu, bw), label=f"bw={bw}")
axes[1].set(title="Effect of bandwidth", xlabel="s", ylabel="φ(s)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Bump support (±3σ):  [{mu - 3*bw_signal}, {mu + 3*bw_signal}]")
print(f"Bump peak value: {phi.max():.3f}")


## 2 · Noise generation pipeline

$$w_i(s) \sim \mathcal{N}(0,1) \xrightarrow{\text{convolve with }g} \tilde{e}_i(s) \xrightarrow{\div\,\text{SD}} e_i(s),\quad \text{SD}(e_i)=1$$


In [ ]:
def make_gaussian_kernel(sd, truncate=4):
    radius = int(truncate * sd)
    u = np.arange(-radius, radius + 1)
    kernel = np.exp(-(u ** 2) / (2 * sd ** 2))
    return u, kernel / kernel.sum()

_, kernel = make_gaussian_kernel(noise_kernel_sd)

# show one noise instance
w_ex = np.random.normal(0, 1, n_pixels)
e_raw = np.convolve(w_ex, kernel, mode="same")
e_ex  = e_raw / e_raw.std()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(s, w_ex, lw=0.8); axes[0].set(title="White noise  w(s)", xlabel="s"); axes[0].grid(True, alpha=0.3)
axes[1].plot(s, e_raw, lw=0.8); axes[1].set(title="After convolution (before rescale)", xlabel="s"); axes[1].grid(True, alpha=0.3)
axes[2].plot(s, e_ex, lw=0.8); axes[2].set(title="Normalised noise  e(s)  SD=1", xlabel="s"); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Noise SD after normalisation: {e_ex.std():.6f}")


## 3 · Generate the full dataset

For each subject $i$:

$$A_i\sim\mathcal{N}(0,\sigma_A^2), \qquad Y_i(s)=A_i\,\phi(s)+e_i(s)$$

Outputs: **Y** (1000 × 512) and **A** (1000,).


In [ ]:
def generate_dataset(n_instances, n_pixels, mu, bw_signal, noise_kernel_sd, sigma_A):
    s = np.arange(n_pixels)
    phi = gaussian_unnormalized(s, mu, bw_signal)
    _, kernel = make_gaussian_kernel(noise_kernel_sd)

    A = np.random.normal(0, sigma_A, size=n_instances)
    Y = np.zeros((n_instances, n_pixels))
    noise_mat = np.zeros((n_instances, n_pixels))

    for i in range(n_instances):
        w = np.random.normal(0, 1, n_pixels)
        e = np.convolve(w, kernel, mode="same")
        e = e / e.std()
        noise_mat[i] = e
        Y[i] = A[i] * phi + e

    return A, Y, noise_mat, phi

A, Y, noise_mat, phi_true = generate_dataset(
    n_instances, n_pixels, mu, bw_signal, noise_kernel_sd, sigma_A)

print(f"Y shape : {Y.shape}   (subjects × pixels)")
print(f"A shape : {A.shape}")
print(f"A mean  : {A.mean():.4f}   A SD: {A.std():.4f}")
print(f"E[Y] across subjects at bump centre: {Y[:, mu].mean():.4f}  (expect ≈0)")


In [ ]:
# ── Sanity-check plots ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) several individual traces coloured by |A|
sort_idx = np.argsort(np.abs(A))
for rank, i in enumerate(sort_idx[[0, 200, 500, 800, 999]]):
    axes[0].plot(s, Y[i], alpha=0.8, lw=0.9, label=f"A={A[i]:.2f}")
axes[0].set(title="Sample subjects (coloured by |A|)", xlabel="s", ylabel="Y(s)")
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# 2) mean of Y across subjects  (should ≈ 0 everywhere since E[A]=0)
axes[1].plot(s, Y.mean(axis=0), label="mean(Y)")
axes[1].plot(s, phi_true, ls="--", label="φ(s)", lw=1.5)
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set(title="Cross-subject mean of Y vs true φ(s)", xlabel="s")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# 3) amplitude histogram
axes[2].hist(A, bins=40, alpha=0.7, edgecolor="white", lw=0.4)
axes[2].axvline(0, color="black", lw=1)
axes[2].set(title=f"Amplitude distribution  σ_A={sigma_A}", xlabel="A", ylabel="count")
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


---
## 4 · Direct OLS regression — pixel-wise (no CNN)

### Why this works

The generative model is $Y_i(s) = A_i\,\phi(s)+e_i(s)$. Regressing $Y[:,s]$ on $A$ at every pixel $s$ gives

$$\hat{\beta}(s) = \frac{A^\top Y[:,s]}{A^\top A} \xrightarrow{n\to\infty} \phi(s)$$

because $\operatorname{Cov}(A, Y(s)) = \sigma_A^2\,\phi(s)$ and the noise is uncorrelated with $A$.

### Regression without intercept

Because $E[A]=0$ and $E[e_i(s)]=0$, the model is zero-mean and no intercept is needed.  
The no-intercept OLS estimator has a one-line closed form: $\hat{\beta} = (A^\top Y) / (A^\top A)$.

### Connection to real brain data

| Simulation | Real neuroimaging |
|---|---|
| $Y_i(s)$ — 1-D signal | Gray-matter intensity at voxel $s$ |
| $A_i$ — amplitude | Subject age |
| $\phi(s)$ — bump shape | Brain regions that change with age |
| $\hat{\beta}(s)$ — coefficient map | Voxel-wise age-effect map |


In [ ]:
# ── OLS: vectorised closed form ───────────────────────────
# beta_hat[s] = (A @ Y[:, s]) / (A @ A)    for all s at once
ATA  = A @ A                          # scalar
ATY  = A @ Y                          # shape (512,)
beta_hat = ATY / ATA                  # shape (512,)  — estimated φ

# ── Residuals & variance ──────────────────────────────────
Y_hat   = np.outer(A, beta_hat)       # shape (1000, 512)
resid   = Y - Y_hat                   # shape (1000, 512)
n, p    = Y.shape
df      = n - 1                       # df = n - 1  (one estimated parameter per pixel, no intercept)
sigma2  = (resid ** 2).sum(axis=0) / df   # residual variance per pixel

# ── SE of beta_hat ────────────────────────────────────────
se_beta = np.sqrt(sigma2 / ATA)       # shape (512,)

# ── t-statistics and two-sided p-values ──────────────────
t_stat  = beta_hat / se_beta          # shape (512,)
p_vals  = 2 * stats.t.sf(np.abs(t_stat), df=df)   # two-sided

print(f"beta_hat range : [{beta_hat.min():.4f}, {beta_hat.max():.4f}]")
print(f"t_stat range   : [{t_stat.min():.2f}, {t_stat.max():.2f}]")
print(f"Min p-value    : {p_vals.min():.2e}")


In [ ]:
# ── Multiple testing correction ───────────────────────────
alpha = 0.05

# Bonferroni
reject_bonf = p_vals < (alpha / n_pixels)

# Benjamini–Hochberg FDR
reject_bh, _, _, _ = multipletests(p_vals, alpha=alpha, method="fdr_bh")

print(f"Significant pixels (Bonferroni, α={alpha}): {reject_bonf.sum()} / {n_pixels}")
print(f"Significant pixels (BH-FDR,    α={alpha}): {reject_bh.sum()} / {n_pixels}")
print(f"True bump support (±3σ, width≈{6*bw_signal}): pixels {mu-3*bw_signal}–{mu+3*bw_signal}")


In [ ]:
# ── Main OLS result figure ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# Panel 1 — estimated vs true shape
axes[0].plot(s, phi_true, color="black", lw=1.5, ls="--", label="True φ(s)")
axes[0].plot(s, beta_hat, color="steelblue", lw=1.5, label="β̂(s)  [OLS estimate]")
axes[0].fill_between(s, beta_hat - 1.96*se_beta, beta_hat + 1.96*se_beta,
                     alpha=0.2, color="steelblue", label="±1.96 SE")
axes[0].set(ylabel="Coefficient", title="OLS estimate β̂(s) vs true signal φ(s)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Panel 2 — t-statistics
axes[1].plot(s, t_stat, color="darkorange", lw=1, label="t(s)")
axes[1].axhline( stats.t.ppf(1 - alpha/(2*n_pixels), df=df), color="red",   ls="--", lw=1, label="Bonferroni threshold")
axes[1].axhline( stats.t.ppf(1 - 0.5*p_vals[reject_bh].max(), df=df) if reject_bh.any() else 0,
                color="purple", ls=":", lw=1, label="BH-FDR threshold (approx)")
axes[1].set(ylabel="t-statistic", title="Pixel-wise t-statistics")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Panel 3 — significance map
axes[2].fill_between(s, reject_bh.astype(float),    alpha=0.5, color="purple", label="BH-FDR significant")
axes[2].fill_between(s, reject_bonf.astype(float),  alpha=0.5, color="red",    label="Bonferroni significant")
axes[2].plot(s, phi_true / phi_true.max(), color="black", ls="--", lw=1, label="True φ(s) [normalised]")
axes[2].set(xlabel="Pixel s", ylabel="Significant (0/1)",
            title="Significant pixels vs true bump support")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# ── Quantitative accuracy: IMSE ───────────────────────────
IMSE_full = np.mean((beta_hat - phi_true) ** 2)

# Restrict to the bump support region (±3σ)
bump_mask = np.abs(s - mu) <= 3 * bw_signal
IMSE_bump = np.mean((beta_hat[bump_mask] - phi_true[bump_mask]) ** 2)

# False-positive rate outside the bump
flat_mask = ~bump_mask
fp_bonf = reject_bonf[flat_mask].mean()
fp_bh   = reject_bh[flat_mask].mean()

print("── OLS estimation accuracy ───────────────────────────")
print(f"IMSE (all pixels)   : {IMSE_full:.6f}")
print(f"IMSE (bump region)  : {IMSE_bump:.6f}")
print(f"Correlation β̂ vs φ  : {np.corrcoef(beta_hat, phi_true)[0,1]:.6f}")
print()
print("── Multiple testing performance ──────────────────────")
print(f"True positive rate  (Bonferroni): {reject_bonf[bump_mask].mean():.3f}")
print(f"True positive rate  (BH-FDR)    : {reject_bh[bump_mask].mean():.3f}")
print(f"False positive rate (Bonferroni): {fp_bonf:.4f}  (nominal α/n = {alpha/n_pixels:.4f})")
print(f"False positive rate (BH-FDR)    : {fp_bh:.4f}  (nominal α = {alpha})")


---
## 5 · OLS sensitivity analysis — how accuracy changes with parameters

We vary one parameter at a time and track:
- **IMSE** — integrated mean squared error between $\hat\beta(s)$ and $\phi(s)$
- **True positive rate (TPR)** — fraction of bump pixels declared significant (power)
- **False positive rate (FPR)** — fraction of flat pixels declared significant (type-I error)

Three sweeps:
1. **Signal amplitude $\sigma_A$** — controls SNR
2. **Bump width $bw\_signal$** — controls spatial extent of the effect
3. **Noise smoothness $noise\_kernel\_sd$** — controls how similar noise looks to signal


In [ ]:
def run_ols(Y, A, phi_true, mu, bw_signal, alpha=0.05):
    """Run pixel-wise OLS and return summary metrics."""
    n, p = Y.shape
    ATA  = A @ A
    beta_hat = (A @ Y) / ATA
    Y_hat = np.outer(A, beta_hat)
    resid = Y - Y_hat
    sigma2 = (resid**2).sum(axis=0) / (n - 1)
    se = np.sqrt(sigma2 / ATA)
    t_stat = beta_hat / se
    p_vals = 2 * stats.t.sf(np.abs(t_stat), df=n-1)
    reject, _, _, _ = multipletests(p_vals, alpha=alpha, method="fdr_bh")
    bump_mask = np.abs(np.arange(p) - mu) <= 3 * bw_signal
    IMSE = np.mean((beta_hat - phi_true)**2)
    TPR  = reject[bump_mask].mean()
    FPR  = reject[~bump_mask].mean()
    return beta_hat, t_stat, reject, IMSE, TPR, FPR

# ── Sweep 1: sigma_A (SNR) ────────────────────────────────
sigma_A_vals = [0.25, 0.5, 1.0, 2.0, 4.0]
results_snr = []
for sa in sigma_A_vals:
    Ai, Yi, _, phi_i = generate_dataset(n_instances, n_pixels, mu, bw_signal, noise_kernel_sd, sa)
    _, _, _, imse, tpr, fpr = run_ols(Yi, Ai, phi_i, mu, bw_signal)
    results_snr.append((sa, imse, tpr, fpr))

# ── Sweep 2: bump width ───────────────────────────────────
bw_signal_vals = [5, 10, 20, 40, 80]
results_bw = []
for bw in bw_signal_vals:
    Ai, Yi, _, phi_i = generate_dataset(n_instances, n_pixels, mu, bw, noise_kernel_sd, sigma_A)
    _, _, _, imse, tpr, fpr = run_ols(Yi, Ai, phi_i, mu, bw)
    results_bw.append((bw, imse, tpr, fpr))

# ── Sweep 3: noise smoothness ─────────────────────────────
nk_vals = [1, 3, 5, 10, 20]
results_nk = []
for nk in nk_vals:
    Ai, Yi, _, phi_i = generate_dataset(n_instances, n_pixels, mu, bw_signal, nk, sigma_A)
    _, _, _, imse, tpr, fpr = run_ols(Yi, Ai, phi_i, mu, bw_signal)
    results_nk.append((nk, imse, tpr, fpr))

print("Sensitivity sweeps complete.")


In [ ]:
# ── Sensitivity plots ─────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
sweeps = [
    (results_snr, sigma_A_vals,    "σ_A (amplitude)",          "Sweep 1: signal amplitude (SNR)"),
    (results_bw,  bw_signal_vals,  "bw_signal (bump width)",   "Sweep 2: bump width"),
    (results_nk,  nk_vals,         "noise_kernel_sd",           "Sweep 3: noise smoothness"),
]
metrics = ["IMSE", "True positive rate (power)", "False positive rate"]
colours = ["steelblue", "darkorange", "crimson"]

for col, (res, xvals, xlabel, title) in enumerate(sweeps):
    vals = np.array(res)
    for row, (metric, col_idx) in enumerate(zip(metrics, [1, 2, 3])):
        ax = axes[row][col]
        ax.plot(xvals, vals[:, col_idx], "o-", color=colours[col], lw=2, ms=6)
        if row == 0:
            ax.set_title(title, fontsize=11)
        if col == 0:
            ax.set_ylabel(metric, fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        if row == 2:    # FPR reference line
            ax.axhline(0.05, color="gray", ls="--", lw=0.8, label="α=0.05")
            ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle("OLS sensitivity: one parameter varied at a time", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Print summary table ───────────────────────────────────
print("=== Sweep 1: σ_A (amplitude / SNR) ===")
print(f"{'σ_A':>8} {'IMSE':>10} {'TPR':>8} {'FPR':>8}")
for sa, imse, tpr, fpr in results_snr:
    print(f"{sa:>8.2f} {imse:>10.5f} {tpr:>8.3f} {fpr:>8.4f}")

print()
print("=== Sweep 2: bw_signal (bump width) ===")
print(f"{'bw':>8} {'IMSE':>10} {'TPR':>8} {'FPR':>8}")
for bw, imse, tpr, fpr in results_bw:
    print(f"{bw:>8} {imse:>10.5f} {tpr:>8.3f} {fpr:>8.4f}")

print()
print("=== Sweep 3: noise_kernel_sd ===")
print(f"{'nk_sd':>8} {'IMSE':>10} {'TPR':>8} {'FPR':>8}")
for nk, imse, tpr, fpr in results_nk:
    print(f"{nk:>8} {imse:>10.5f} {tpr:>8.3f} {fpr:>8.4f}")


---
## 6 · Multiscale analysis — pre-smooth Y, then regress

### Idea

Pre-convolve each subject's observed signal $Y_i(s)$ with a Gaussian kernel of bandwidth $h$:

$$Y_i^{(h)}(s) = (Y_i * g_h)(s)$$

then run the same pixel-wise OLS on $Y^{(h)}$. 

### Why it helps (matched-filter intuition)

Pre-smoothing at scale $h$ is equivalent to projecting both signal and noise onto functions that are smooth at scale $h$.  
- When $h \approx bw\_signal$: the kernel acts as a **matched filter** — it resonates with the bump shape and suppresses noise that varies faster than the bump. SNR improves.  
- When $h \ll bw\_signal$: little change — the signal is already smooth relative to $h$.  
- When $h \gg bw\_signal$: the bump gets blurred out and you lose spatial resolution.

### Important: re-normalise after smoothing

After convolving $Y$ with $g_h$, the noise variance changes (it decreases).  
We **must** re-normalise $Y^{(h)}$ before computing OLS t-statistics, otherwise the SE estimates are wrong.  
We normalise by the theoretical noise SD after convolution: $\text{SD}(e^{(h)}) = \|g_h\|_2$ (since $e$ started at SD=1).


In [ ]:
def kernel_l2_norm(sd, truncate=4):
    """L2 norm of the Gaussian kernel — used to rescale noise SD after convolution."""    _, k = make_gaussian_kernel(sd, truncate)
    return np.sqrt((k**2).sum())

def multiscale_smooth(Y, h):
    """
    Pre-smooth every subject's Y(s) with a Gaussian kernel of SD=h.
    Returns Y_h (same shape as Y) with noise SD adjusted back to 1.
    """
    _, k = make_gaussian_kernel(h)
    Y_smooth = np.array([np.convolve(Y[i], k, mode="same") for i in range(Y.shape[0])])
    # After convolving noise (SD=1) with kernel k, noise SD becomes ||k||_2
    # Dividing by ||k||_2 restores noise SD to 1
    Y_smooth /= kernel_l2_norm(h)
    return Y_smooth

# Try a range of pre-smoothing bandwidths
h_vals = [1, 5, 10, 20, 40, 80]

multiscale_results = []
beta_hats_ms = {}

for h in h_vals:
    Y_h = multiscale_smooth(Y, h)
    b_h, t_h, rej_h, imse_h, tpr_h, fpr_h = run_ols(Y_h, A, phi_true, mu, bw_signal)
    multiscale_results.append((h, imse_h, tpr_h, fpr_h))
    beta_hats_ms[h] = b_h

# Baseline: no pre-smoothing (h→0, equivalent to h=1 pixel, essentially raw)
_, _, _, imse_raw, tpr_raw, fpr_raw = run_ols(Y, A, phi_true, mu, bw_signal)
print("No pre-smoothing (baseline):")
print(f"  IMSE={imse_raw:.5f}  TPR={tpr_raw:.3f}  FPR={fpr_raw:.4f}")
print()
print(f"{'h':>6} {'IMSE':>10} {'TPR':>8} {'FPR':>8}")
for h, imse, tpr, fpr in multiscale_results:
    print(f"{h:>6} {imse:>10.5f} {tpr:>8.3f} {fpr:>8.4f}")


In [ ]:
# ── Plot 1: estimated β̂(s) for each pre-smoothing bandwidth ──
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, h in zip(axes, h_vals):
    ax.plot(s, phi_true, color="black", lw=1.5, ls="--", label="True φ(s)")
    ax.plot(s, beta_hats_ms[h], color="steelblue", lw=1.5, label=f"β̂(s)  h={h}")
    ax.set_title(f"Pre-smoothing  h={h}", fontsize=11)
    ax.set(xlabel="s", ylabel="coefficient")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("OLS coefficient maps under multiscale pre-smoothing", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 2: IMSE / TPR / FPR vs h ────────────────────────
h_arr  = np.array([r[0] for r in multiscale_results])
imse_arr = np.array([r[1] for r in multiscale_results])
tpr_arr  = np.array([r[2] for r in multiscale_results])
fpr_arr  = np.array([r[3] for r in multiscale_results])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].plot(h_arr, imse_arr, "o-", color="steelblue", lw=2, ms=7)
axes[0].axhline(imse_raw, color="gray", ls="--", lw=1, label="No smoothing baseline")
axes[0].axvline(bw_signal, color="green", ls=":", lw=1.5, label=f"bw_signal={bw_signal}")
axes[0].set(title="IMSE vs pre-smoothing bandwidth h",
            xlabel="h (pre-smoothing SD)", ylabel="IMSE")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(h_arr, tpr_arr, "o-", color="darkorange", lw=2, ms=7)
axes[1].axhline(tpr_raw, color="gray", ls="--", lw=1, label="No smoothing")
axes[1].axvline(bw_signal, color="green", ls=":", lw=1.5, label=f"bw_signal={bw_signal}")
axes[1].set(title="True positive rate (power) vs h",
            xlabel="h (pre-smoothing SD)", ylabel="TPR")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(h_arr, fpr_arr, "o-", color="crimson", lw=2, ms=7)
axes[2].axhline(fpr_raw, color="gray", ls="--", lw=1, label="No smoothing")
axes[2].axhline(0.05, color="black", ls="--", lw=0.8, label="Nominal α=0.05")
axes[2].axvline(bw_signal, color="green", ls=":", lw=1.5, label=f"bw_signal={bw_signal}")
axes[2].set(title="False positive rate vs h",
            xlabel="h (pre-smoothing SD)", ylabel="FPR")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(f"Multiscale analysis  (true bw_signal={bw_signal})", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 3: stacked heatmap of β̂(s) across scales ────────
beta_matrix = np.array([beta_hats_ms[h] for h in h_vals])  # (n_scales, 512)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

im = axes[0].imshow(beta_matrix, aspect="auto",
                    extent=[0, n_pixels, h_vals[-1]+5, h_vals[0]-1],
                    cmap="RdBu_r", origin="upper",
                    vmin=-phi_true.max(), vmax=phi_true.max())
axes[0].set_yticks(h_vals)
axes[0].axvline(mu - 3*bw_signal, color="lime", ls="--", lw=1)
axes[0].axvline(mu + 3*bw_signal, color="lime", ls="--", lw=1, label="±3σ bump support")
axes[0].set(title="β̂(s, h) — coefficient maps across pre-smoothing scales",
            xlabel="Pixel s", ylabel="Pre-smoothing SD  h")
axes[0].legend(fontsize=9)
plt.colorbar(im, ax=axes[0], label="β̂")

# p-value heatmap (significance)
p_matrix = np.zeros_like(beta_matrix)
for row, h in enumerate(h_vals):
    Y_h = multiscale_smooth(Y, h)
    ATA = A @ A
    beta_h = (A @ Y_h) / ATA
    Y_hat_h = np.outer(A, beta_h)
    resid_h = Y_h - Y_hat_h
    sigma2_h = (resid_h**2).sum(axis=0) / (n_instances - 1)
    se_h = np.sqrt(sigma2_h / ATA)
    t_h = beta_h / se_h
    p_h = 2 * stats.t.sf(np.abs(t_h), df=n_instances-1)
    reject_h, _, _, _ = multipletests(p_h, alpha=0.05, method="fdr_bh")
    p_matrix[row] = reject_h.astype(float)

im2 = axes[1].imshow(p_matrix, aspect="auto",
                     extent=[0, n_pixels, h_vals[-1]+5, h_vals[0]-1],
                     cmap="Purples", origin="upper", vmin=0, vmax=1)
axes[1].set_yticks(h_vals)
axes[1].axvline(mu - 3*bw_signal, color="lime", ls="--", lw=1)
axes[1].axvline(mu + 3*bw_signal, color="lime", ls="--", lw=1, label="±3σ bump support")
axes[1].set(title="BH-FDR significant pixels across scales  (purple = significant)",
            xlabel="Pixel s", ylabel="Pre-smoothing SD  h")
axes[1].legend(fontsize=9)
plt.colorbar(im2, ax=axes[1], label="Significant (0/1)")

plt.tight_layout(); plt.show()


---
## 7 · Side-by-side comparison and key takeaways

We compare:
- **OLS (no smoothing)** — the direct baseline
- **Multiscale OLS at optimal h** — best pre-smoothing scale found above


In [ ]:
# ── Find optimal h (minimises IMSE) ──────────────────────
best_h_idx = np.argmin(imse_arr)
best_h     = h_vals[best_h_idx]
print(f"Optimal pre-smoothing bandwidth: h = {best_h}  (true bw_signal = {bw_signal})")
print(f"IMSE at best h       : {imse_arr[best_h_idx]:.5f}")
print(f"IMSE without smoothing: {imse_raw:.5f}")
print(f"Relative improvement : {100*(imse_raw - imse_arr[best_h_idx])/imse_raw:.1f}%")


In [ ]:
# ── Final summary figure ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Panel 1: coefficient maps
axes[0].plot(s, phi_true,              color="black",      lw=2,   ls="--", label="True φ(s)")
axes[0].plot(s, beta_hat,              color="steelblue",  lw=1.5,          label="OLS β̂(s)  [no smoothing]")
axes[0].plot(s, beta_hats_ms[best_h], color="darkorange", lw=1.5,          label=f"OLS β̂(s)  [h={best_h}, optimal]")
axes[0].set(ylabel="Estimated coefficient",
            title="Comparison: OLS vs Multiscale OLS")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Panel 2: significance maps for each method
Y_opt = multiscale_smooth(Y, best_h)
b_opt = (A @ Y_opt) / (A @ A)
resid_opt = Y_opt - np.outer(A, b_opt)
sigma2_opt = (resid_opt**2).sum(axis=0) / (n_instances - 1)
se_opt = np.sqrt(sigma2_opt / (A @ A))
t_opt = b_opt / se_opt
p_opt = 2 * stats.t.sf(np.abs(t_opt), df=n_instances-1)
rej_opt, _, _, _ = multipletests(p_opt, alpha=0.05, method="fdr_bh")

axes[1].fill_between(s, reject_bh.astype(float),  alpha=0.4, color="steelblue",  label=f"OLS no smooth  (TPR={tpr_raw:.2f}, FPR={fpr_raw:.3f})")
axes[1].fill_between(s, rej_opt.astype(float),     alpha=0.4, color="darkorange", label=f"OLS h={best_h}    (TPR={tpr_arr[best_h_idx]:.2f}, FPR={fpr_arr[best_h_idx]:.3f})")
axes[1].plot(s, phi_true / phi_true.max(), color="black", ls="--", lw=1, label="True φ(s) [normalised]")
axes[1].set(xlabel="Pixel s", ylabel="Significant (0/1)",
            title="BH-FDR significant pixels (α=0.05)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


---
## 8 · Summary and interpretation

| Method | IMSE | TPR (power) | FPR | Notes |
|---|---|---|---|---|
| OLS, no pre-smoothing | — | — | — | baseline |
| Multiscale OLS, optimal h | — | — | — | best scale ≈ bw_signal |

**Key lessons for the real brain-age project:**

1. **OLS on a known scalar (A / age) directly recovers the signal shape** — this is why voxel-wise regression is the standard in neuroimaging.  
2. **Pre-smoothing at the right scale improves power** — when bump width ≈ noise smoothness, matched-filter pre-smoothing helps. In practice, brain images are typically pre-smoothed with a 6–8 mm FWHM kernel for exactly this reason.  
3. **Too much smoothing blurs the bump and destroys localisation** — there is an optimal scale close to the true effect size.  
4. **BH-FDR controls the expected proportion of false positives** reliably across all settings tested here.  
5. **These OLS results are the ground-truth reference for the CNN + AGOP analysis to come** — if the CNN's saliency map does not match the OLS coefficient map, it is not recovering the signal correctly.
